In [ ]:
import math
import csv
import os

from google.colab import files

print("\nUpload PERFECT Abaqus .inp file.")

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No input file was uploaded.")


input_file = list(uploaded.keys())[0]

print("\nInput file selected:")
print(input_file)

print("\n" + "=" * 65)
print("ENTER IMPERFECTION PARAMETERS")
print("=" * 65)

A = float(
    input(
        "\nEnter imperfection amplitude A [mm]: "
    )
)

m = int(
    input(
        "Enter circumferential wave number m: "
    )
)

beta = int(
    input(
        "Enter longitudinal half-wave number beta "
        "(odd integer): "
    )
)

theta0_deg = float(
    input(
        "Enter phase angle theta0 [degrees] "
        "(enter 0 for initial case): "
    )
)



if A < 0:
    raise ValueError(
        "Imperfection amplitude A must be >= 0."
    )

if m <= 0:
    raise ValueError(
        "Circumferential wave number m must be a positive integer."
    )

if beta <= 0:
    raise ValueError(
        "Beta must be a positive integer."
    )


if beta % 2 == 0:
    raise ValueError(
        "For the Shen-type periodic formulation used in "
        "your report, beta must be odd: 1, 3, 5, ..."
    )



theta0 = math.radians(theta0_deg)


with open(
    input_file,
    "r"
) as f:

    lines = f.readlines()

instance_start = None
instance_end = None

for i, line in enumerate(lines):

    line_upper = line.strip().upper()

    if line_upper.startswith("*INSTANCE"):

        instance_start = i
        break


if instance_start is None:

    raise RuntimeError(
        "No *INSTANCE block was found in the input file."
    )


for i in range(
    instance_start + 1,
    len(lines)
):

    if (
        lines[i]
        .strip()
        .upper()
        .startswith("*END INSTANCE")
    ):

        instance_end = i
        break


if instance_end is None:

    raise RuntimeError(
        "No *END INSTANCE found."
    )


print("\nFirst Abaqus instance detected.")


node_start = None
node_end = None

for i in range(
    instance_start + 1,
    instance_end
):

    if (
        lines[i]
        .strip()
        .upper()
        .startswith("*NODE")
    ):

        node_start = i
        break


if node_start is None:

    raise RuntimeError(
        "No *NODE block was found inside the first instance."
    )


node_end = node_start + 1

while node_end < instance_end:

    line = lines[node_end].strip()

    if line.startswith("*"):
        break

    node_end += 1


print("Node block detected.")


def parse_node(line):

    """
    Reads a standard Abaqus node line:

    node_id, x, y, z

    Returns:
        node_id, x, y, z

    Returns None if the line is not a valid node.
    """

    parts = line.strip().split(",")

    if len(parts) < 4:
        return None

    try:

        node_id = int(
            parts[0].strip()
        )

        x = float(
            parts[1].strip()
        )

        y = float(
            parts[2].strip()
        )

        z = float(
            parts[3].strip()
        )

    except ValueError:

        return None

    return node_id, x, y, z


nodes = []

for line in lines[
    node_start + 1:
    node_end
]:

    parsed = parse_node(line)

    if parsed is not None:

        nodes.append(parsed)


if len(nodes) == 0:

    raise RuntimeError(
        "No valid nodes were found."
    )


print(
    "\nTotal nodes found in first instance:",
    len(nodes)
)


radial_values = []

for node_id, x, y, z in nodes:

    r = math.sqrt(
        x**2 + y**2
    )

    radial_values.append(r)


Rm = sum(
    radial_values
) / len(
    radial_values
)


z_values = [
    node[3]
    for node in nodes
]

z_min = min(z_values)
z_max = max(z_values)

L = z_max - z_min


if L <= 0:

    raise RuntimeError(
        "Detected member length L <= 0."
    )



print("DETECTED GEOMETRY")


print(
    "Mean radius Rm       =",
    Rm,
    "mm"
)

print(
    "Minimum Z            =",
    z_min,
    "mm"
)

print(
    "Maximum Z            =",
    z_max,
    "mm"
)

print(
    "Member length L      =",
    L,
    "mm"
)



print("IMPERFECTION PARAMETERS")


print(
    "Amplitude A          =",
    A,
    "mm"
)

print(
    "Circumferential m    =",
    m
)

print(
    "Longitudinal beta    =",
    beta
)

print(
    "Phase theta0         =",
    theta0_deg,
    "degrees"
)

print(
    "Phase theta0         =",
    theta0,
    "radians"
)

print(
    "Longitudinal L       =",
    L,
    "mm"
)

print(
    "\nImperfection equation:"
)

print(
    "Delta r(theta,z) = "
    "A*cos[m(theta-theta0)]*"
    "cos[beta*pi(z/L - 1/2)]"
)


output_filename = (
    os.path.splitext(input_file)[0]
    +
    "_imperfect.inp"
)

csv_filename = (
    os.path.splitext(input_file)[0]
    +
    "_imperfection.csv"
)


new_node_lines = []

csv_rows = []

max_delta = -float("inf")
min_delta = float("inf")

modified_nodes = 0


for line in lines[
    node_start + 1:
    node_end
]:

    parsed = parse_node(line)


    if parsed is None:

        new_node_lines.append(line)

        continue


    node_id, x, y, z = parsed

    r = math.sqrt(
        x**2 + y**2
    )

    theta = math.atan2(
        y,
        x
    )


    z_local = z - z_min



    delta_r = (
        A
        *
        math.cos(
            m * (
                theta - theta0
            )
        )
        *
        math.cos(
            beta
            * math.pi
            * (
                z_local / L
                - 0.5
            )
        )
    )



    r_new = Rm + delta_r



    x_new = (
        r_new
        *
        math.cos(theta)
    )

    y_new = (
        r_new
        *
        math.sin(theta)
    )

    z_new = z



    new_node_lines.append(
        "{}, {:.12g}, {:.12g}, {:.12g}\n".format(
            node_id,
            x_new,
            y_new,
            z_new
        )
    )



    csv_rows.append([
        node_id,

        x,
        y,
        z,

        r,
        theta,

        z_local,

        delta_r,

        r_new,

        x_new,
        y_new,
        z_new
    ])



    modified_nodes += 1

    max_delta = max(
        max_delta,
        delta_r
    )

    min_delta = min(
        min_delta,
        delta_r
    )



output_lines = (
    lines[
        :node_start + 1
    ]
    +
    new_node_lines
    +
    lines[
        node_end:
    ]
)



with open(
    output_filename,
    "w"
) as f:

    f.writelines(
        output_lines
    )


with open(
    csv_filename,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)


    writer.writerow([
        "Node_ID",

        "x_original",
        "y_original",
        "z_original",

        "r_original",
        "theta_rad",
        "z_local",

        "Delta_r",

        "r_new",

        "x_new",
        "y_new",
        "z_new"
    ])


    writer.writerows(
        csv_rows
    )



lambda_theta = (
    2.0
    * math.pi
    * Rm
    / m
)



lambda_z = (
    2.0
    * L
    / beta
)



print("\n")

print("IMPERFECTION GENERATION COMPLETE")


print("\nINPUT")
print(
    "Perfect input file:",
    input_file
)

print("\nOUTPUT")
print(
    "Modified input file:",
    output_filename
)

print(
    "CSV file:",
    csv_filename
)

print("\nGEOMETRY")
print(
    "Mean radius Rm:",
    Rm,
    "mm"
)

print(
    "Member length L:",
    L,
    "mm"
)

print("\nIMPERFECTION")
print(
    "A:",
    A,
    "mm"
)

print(
    "m:",
    m
)

print(
    "beta:",
    beta
)

print(
    "theta0:",
    theta0_deg,
    "degrees"
)

print(
    "Maximum Delta r:",
    max_delta,
    "mm"
)

print(
    "Minimum Delta r:",
    min_delta,
    "mm"
)

print(
    "Maximum |Delta r|:",
    max(
        abs(max_delta),
        abs(min_delta)
    ),
    "mm"
)

print("\nWAVELENGTHS")

print(
    "Circumferential wavelength lambda_theta:",
    lambda_theta,
    "mm"
)

print(
    "Longitudinal wavelength lambda_z:",
    lambda_z,
    "mm"
)

print("\nNODES")
print(
    "Modified nodes:",
    modified_nodes
)




print("\nDownloading files...")

files.download(
    output_filename
)

files.download(
    csv_filename
)

print("\nFinished.")


Upload PERFECT Abaqus .inp file.


Saving Job-1.inp to Job-1 (12).inp

Input file selected:
Job-1 (12).inp

ENTER IMPERFECTION PARAMETERS

Enter imperfection amplitude A [mm]: 10
Enter circumferential wave number m: 5
Enter longitudinal half-wave number beta (odd integer): 5
Enter phase angle theta0 [degrees] (enter 0 for initial case): 0

First Abaqus instance detected.
Node block detected.

Total nodes found in first instance: 32399
DETECTED GEOMETRY
Mean radius Rm       = 68.06000007029232 mm
Minimum Z            = 0.0 mm
Maximum Z            = 420.279999 mm
Member length L      = 420.279999 mm
IMPERFECTION PARAMETERS
Amplitude A          = 10.0 mm
Circumferential m    = 5
Longitudinal beta    = 5
Phase theta0         = 0.0 degrees
Phase theta0         = 0.0 radians
Longitudinal L       = 420.279999 mm

Imperfection equation:
Delta r(theta,z) = A*cos[m(theta-theta0)]*cos[beta*pi(z/L - 1/2)]


IMPERFECTION GENERATION COMPLETE

INPUT
Perfect input file: Job-1 (12).inp

OUTPUT
Modified input file: Job-1 (12)_imperfect.i

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Finished.
